# Enhanced Green-Apple Detection, Relative Depth, and Duplicate-Safe Tracking

This notebook follows the requested workflow: **80-FPS timeline → 800 uniform frames → augmentation → grouped 70/10/20 split → automatic green-apple labels → enhanced YOLOv8 (AAM, EBCA, FM, KWConv) → AdaBins relative depth → depth-aware ByteTrack counting**.

The input is 59.94 FPS. The 80-FPS stage resamples timestamps and records repeated source-frame indices rather than fabricating new camera content. Generated data is local under `Dataset/` and `Results/`; it is excluded from Git.

## How to run

Run cells in numbered order. The full dataset, automatic labeling, training, and inference cells have `RUN_...` flags so they write files only after you enable them. The full training configuration uses `patience=0`, which completes all requested epochs.

In [ ]:
# %% 01) Imports and project paths
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Codeing' / 'fruit_pipeline.py').exists():
    PROJECT_ROOT = Path('/home/hari/Desktop/AAM, EBCA, FM, KWConv').resolve()
if str(PROJECT_ROOT / 'Codeing') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'Codeing'))

import fruit_pipeline as fp

paths = fp.project_paths(PROJECT_ROOT)
paths.ensure_layout()
print('Project root:', paths.root)
print('Input video :', paths.raw_video)
print('Dataset dir :', paths.dataset)
print('Results dir :', paths.results)


## Runtime and GPU check

Records installed PyTorch, CUDA availability, NVIDIA driver response, and the selected device. Dataset preparation can use CPU; full YOLOv8x training is protected until a CUDA-enabled PyTorch build and working NVIDIA driver are available.

In [ ]:
# %% 02) Detect NVIDIA/CUDA and choose the execution device
runtime = fp.hardware_report(paths)
print(json.dumps(runtime, indent=2))
DEVICE = runtime['recommended_device']
if DEVICE == 'cpu':
    print('CUDA is not usable. Prepare data on CPU, then repair CUDA before full YOLOv8x training.')
else:
    print(f'Using GPU {DEVICE}: {runtime.get("cuda_device")}')


## Video preflight and 80-FPS accounting

Inspects the source without writing frames. It reports real frame rate, dimensions, duration, source count, and requested 80-FPS timeline length. This video produces 5,105 output timestamps over about 63.814 seconds.

In [ ]:
# %% 03) Inspect the video and calculate the 80-FPS timeline
TARGET_FPS = 80.0
preflight = fp.video_preflight(paths.raw_video, target_fps=TARGET_FPS)
print(json.dumps(preflight, indent=2))
assert preflight['playable'], preflight.get('error', 'Input video is not playable.')
assert preflight['requested_80fps_timeline_frames'] == 5105


## Stage 1 — documented 80-FPS extraction

Writes 5,105 frames into `Dataset/frames/80fps_timeline/` and an `80fps_timeline.jsonl` manifest. The MP4 is read-only; only generated `timeline_*.jpg` files are replaceable when rebuild is enabled.

In [ ]:
# %% 04) Extract the 80-FPS timeline
RUN_80FPS_EXTRACTION = False  # Set True to write the 5,105 local timeline images.
REBUILD_80FPS_TIMELINE = False

if RUN_80FPS_EXTRACTION:
    timeline_manifest = fp.extract_80fps_timeline(paths, target_fps=TARGET_FPS, overwrite=REBUILD_80FPS_TIMELINE)
    print(f'80-FPS timeline frames written: {len(timeline_manifest):,}')
    print('Manifest:', paths.manifests / '80fps_timeline.jsonl')
else:
    print('Not run. Enable after reviewing Cell 03.')


## Stage 2 — exact uniform sample of 800

Selects exactly 800 evenly spaced timeline positions and copies them to `Dataset/frames/picked_800_uniform/`. A manifest preserves the selected timeline position and filename for every picked image.

In [ ]:
# %% 05) Uniformly select 800 frames
RUN_UNIFORM_SELECTION = False  # Set True after Cell 04 succeeds.
REBUILD_UNIFORM_SELECTION = False

if RUN_UNIFORM_SELECTION:
    picked_manifest = fp.pick_uniform_800(paths, count=800, overwrite=REBUILD_UNIFORM_SELECTION)
    print(f'Uniform frames picked: {len(picked_manifest):,}')
    for row in picked_manifest[:5]:
        print(row)
else:
    print('Not run. Requires the timeline from Cell 04.')


## Stage 3 — augment the 800 picked images

Creates ten documented visual variants per selected image: flips, brightness changes, rotations, small rotation, motion blur, and Gaussian noise. It creates no labels at this stage because automatic labeling is intentionally performed after splitting.

In [ ]:
# %% 06) Create 8,000 augmentation images
RUN_AUGMENTATION = False  # Set True after Cell 05 succeeds.
REBUILD_AUGMENTATION = False

if RUN_AUGMENTATION:
    augmentation_summary = fp.augment_picked_800(paths, seed=42, overwrite=REBUILD_AUGMENTATION)
    print(json.dumps(augmentation_summary, indent=2))
else:
    print('Not run. Requires exactly 800 picked images.')


## Dataset inventory

Reports the exact requested counts of 80-FPS timeline images, uniform 800 selection, augmentations, and their combined total. This is read-only reporting after the generation stages.

In [ ]:
# %% 07) Display all frame counts and manifest locations
def count_images(folder, pattern='*.jpg'):
    return len(list(Path(folder).glob(pattern)))

inventory = {
    '80fps_timeline_frames': count_images(paths.frames_80fps, 'timeline_*.jpg'),
    'uniformly_picked_frames': count_images(paths.picked_800, 'pick_*.jpg'),
    'augmentation_frames': count_images(paths.augmented_800, '*__*.jpg'),
}
inventory['picked_plus_augmentations'] = inventory['uniformly_picked_frames'] + inventory['augmentation_frames']
print(json.dumps(inventory, indent=2))
print('Picked manifest      :', paths.manifests / 'picked_800_uniform.jsonl')
print('Augmentation manifest:', paths.manifests / 'augmentation_manifest.jsonl')


## Stage 4 — grouped 70/10/20 split

This split preserves every original and its ten derivatives in one partition, preventing augmentation leakage. It produces exactly 560/80/160 source-frame groups and 6,160/880/1,760 images in train/validation/test. Labeling occurs only after this split.

In [ ]:
# %% 08) Build the leakage-safe YOLO image split
RUN_GROUPED_SPLIT = False  # Set True after Cell 06 succeeds.
REBUILD_GROUPED_SPLIT = False

if RUN_GROUPED_SPLIT:
    split_summary = fp.build_grouped_dataset_split(paths, train_fraction=0.70, val_fraction=0.10, overwrite=REBUILD_GROUPED_SPLIT)
    print(json.dumps(split_summary, indent=2))
    assert (split_summary['train_groups'], split_summary['val_groups'], split_summary['test_groups']) == (560, 80, 160)
else:
    print('Not run. Requires the 800 originals and all their variants.')


## Stage 5 — automatic single-class green-apple labels

No manual labels are used. A COCO-pretrained YOLOv8 detector proposes the COCO `apple` category from full-frame and overlapping tiled inference. NMS removes duplicate detections and every accepted box is saved as class `green_apple`; previews and an audit manifest are generated. These are pseudo-labels, so they cannot prove a real >90% result without independent ground-truth test labels.

In [ ]:
# %% 09) Automatically label all split images
RUN_AUTOMATIC_LABELING = False  # Set True after Cell 08 succeeds.
REBUILD_AUTOMATIC_LABELS = False
AUTO_LABEL_MODEL = 'yolov8x.pt'
AUTO_LABEL_CONFIDENCE = 0.15
AUTO_LABEL_IMAGE_SIZE = 1280

if RUN_AUTOMATIC_LABELING:
    auto_label_summary = fp.auto_label_after_split(
        paths, model_name=AUTO_LABEL_MODEL, confidence=AUTO_LABEL_CONFIDENCE,
        image_size=AUTO_LABEL_IMAGE_SIZE, device=DEVICE, tiled=True,
        overwrite=REBUILD_AUTOMATIC_LABELS,
    )
    print(json.dumps(auto_label_summary, indent=2))
    print('Labels  :', paths.yolo_labels)
    print('Previews:', paths.auto_previews)
else:
    print('Not run. GPU is strongly recommended for 8,800 tiled automatic-label images.')


## Dataset and label audit

Verifies image/label parity and reports the boxes, empty labels, and the one permitted class. It saves `Results/dataset/dataset_audit.json`. Training refuses to start until this audit passes.

In [ ]:
# %% 10) Audit automatic labels
audit = fp.validate_dataset(paths)
print(json.dumps(audit, indent=2))
if any(part['images'] for part in audit['splits'].values()):
    assert audit['valid'], 'Every split image must have an automatic YOLO label file before training.'
else:
    print('Dataset split and automatic labels have not been created yet; run Cells 04–09 before training.')


## AAM, EBCA, FM, and KWConv

The four implemented PyTorch modules match the submitted architecture: AAM combines colour/shape/scale attention, EBCA exchanges P3/P4 information, FM gates contextual features for occlusion, and KWConv selects mixed receptive fields for scale variation.

In [ ]:
# %% 11) Construct and inspect all custom feature modules
import torch

aam = fp.AAM(channels=128).to(DEVICE).eval()
ebca = fp.EBCA(p3_channels=64, p4_channels=128).to(DEVICE).eval()
fm = fp.FM(channels=128).to(DEVICE).eval()
kwconv = fp.KWConv(channels=128).to(DEVICE).eval()
with torch.inference_mode():
    p3 = torch.zeros(1, 64, 80, 80, device=DEVICE)
    p4 = torch.zeros(1, 128, 40, 40, device=DEVICE)
    print('AAM   :', tuple(aam(p4).shape))
    print('EBCA  :', tuple(shape.shape for shape in ebca(p3, p4)))
    print('FM    :', tuple(fm(p4).shape))
    print('KWConv:', tuple(kwconv(p4).shape))


## Integrated enhanced YOLOv8 feature neck

The proposed neck runs **AAM → EBCA → FM → KWConv** on YOLO P3/P4/P5 before the standard Detect head. The smoke test confirms compatible tensor shapes and writes an architecture record to `Results/architecture/`.

In [ ]:
# %% 12) Verify the integrated AAM–EBCA–FM–KWConv neck
architecture_report = fp.module_smoke_test(paths, device=DEVICE)
print(json.dumps(architecture_report, indent=2))


## Full training run

This uses a one-class enhanced YOLOv8x model, 1,024-pixel images, auto GPU batch sizing, and 100 epochs. The custom trainer inserts the enhanced neck into the model actually being optimized. Results, checkpoints, plots, and curves go to `Results/training/`. `patience=0` prevents an early-stopping interruption.

In [ ]:
# %% 13) Train enhanced YOLOv8x through all requested epochs
RUN_FULL_TRAINING = False  # Set True after Cell 10 passes and CUDA works.
REQUIRE_CUDA_FOR_TRAINING = True
TRAINING_MODEL = 'yolov8x.pt'
TRAINING_EPOCHS = 100
TRAINING_IMAGE_SIZE = 1024
TRAINING_BATCH = -1
TRAINING_WORKERS = 4
TRAINING_NAME = 'aam_ebca_fm_kwconv_green'

if RUN_FULL_TRAINING:
    if REQUIRE_CUDA_FOR_TRAINING and DEVICE == 'cpu':
        raise RuntimeError('CUDA is unavailable. Install a working NVIDIA driver and CUDA-enabled PyTorch, rerun Cell 02, then start the complete run.')
    train_results = fp.train_enhanced_yolov8(
        paths, model_name=TRAINING_MODEL, epochs=TRAINING_EPOCHS,
        image_size=TRAINING_IMAGE_SIZE, batch=TRAINING_BATCH, device=DEVICE,
        workers=TRAINING_WORKERS, run_name=TRAINING_NAME,
    )
    print('Training completed:', train_results)
else:
    print('Not run. Enabling the flag launches all 100 epochs, not a partial training run.')


## Precision, recall, F1, and mAP

After training, validation generates Ultralytics metrics and a compact final report with precision, recall, calculated F1, mAP@50, and mAP@50–95 in `Results/validation/precision_recall_f1_summary.json`. The score is scientifically valid only against independent ground truth, not test pseudo-labels created by the same COCO model.

In [ ]:
# %% 14) Validate trained weights and save final metrics
RUN_FINAL_VALIDATION = False
BEST_WEIGHTS = paths.training_results / TRAINING_NAME / 'weights' / 'best.pt'

if RUN_FINAL_VALIDATION:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}')
    validation_results = fp.validate_model(paths, BEST_WEIGHTS, image_size=TRAINING_IMAGE_SIZE, device=DEVICE)
    metric_summary = fp.summarize_metrics(paths, paths.training_results / TRAINING_NAME)
    print(json.dumps(metric_summary, indent=2))
else:
    print('Not run. It needs:', BEST_WEIGHTS)


## AdaBins relative depth

The optional official AdaBins adapter estimates relative pixel depth. Robust depth per apple box is used to distinguish closer/farther fruit during tracking. Values are not physical metres until camera calibration is supplied.

In [ ]:
# %% 15) Initialize AdaBins for relative-depth estimation
RUN_ADABINS = False
ADABINS_REPOSITORY = PROJECT_ROOT / 'third_party' / 'AdaBins'
adabins = None

if RUN_ADABINS:
    adabins = fp.AdaBinsDepth(ADABINS_REPOSITORY, dataset='kitti', device=str(DEVICE))
    print('AdaBins ready; tracking will use relative depth.')
else:
    print('Optional stage not initialized. Tracking can run without depth.')


## Depth-aware ByteTrack-style tracking and duplicate-safe count

The final stage combines detections with a two-stage confidence association, motion prediction, IoU, and optional depth. Persistent IDs are written frame-by-frame, while a line counter records each confirmed track only once. It outputs annotated video, tracks CSV, and summary in `Results/inference/`.

In [ ]:
# %% 16) Run tracking and save the unique apple count
RUN_TRACKING = False
COUNT_LINE = ((0, 540), (1919, 540))

if RUN_TRACKING:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}')
    tracking_summary = fp.run_depth_aware_tracking(
        paths, weights=BEST_WEIGHTS, adabins=adabins, confidence=0.10,
        image_size=TRAINING_IMAGE_SIZE, device=DEVICE, line=COUNT_LINE,
    )
    print(json.dumps(tracking_summary, indent=2))
else:
    print('Not run. Requires completed training; AdaBins is optional.')


## Final artifact inventory

Lists the generated local data and results after running the relevant cells. The repository contains only the reusable pipeline, notebook, environment configuration, and documentation—not the video, generated images, labels, checkpoints, or results.

In [ ]:
# %% 17) List dataset, results, manifests, and metrics
for folder in (
    paths.frames_80fps, paths.picked_800, paths.augmented_800, paths.yolo,
    paths.dataset_results, paths.architecture_results, paths.training_results,
    paths.validation_results, paths.inference_results,
):
    print(f'{folder.relative_to(paths.root)}: {sum(p.is_file() for p in folder.rglob("*")):,} files')

print('\nReports:')
for report in sorted(paths.manifests.glob('*.jsonl')) + sorted(paths.dataset_results.glob('*.json')) + sorted(paths.validation_results.glob('*.json')):
    print(' -', report.relative_to(paths.root))
